# Watching a SuperPEEC solve live

This notebook launches a solve in a **separate process** and watches it
through the status API — the same mechanism a GUI or any other tool
would use. It shows a live progress line, a Z(f) plot that fills in as
frequency points complete, and a post-mortem of where the time went.

Reference: `docs/status_api.md` (schema 1). Run this notebook from the
`examples/` directory (Jupyter's default when you open it here).
Needs `matplotlib` and `tqdm` (both plain pip installs).


In [ ]:
# -- launch: any SuperPEEC entry point works; the CLI is the simplest.
# The env var SPPEEC_STATUS makes the solver write an atomically
# updated JSON status file; SPPEEC_STATUS_EVENTS adds a JSONL
# transition log. Neither changes any converged number.
import os
import subprocess
import sys
import tempfile

ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'src'))
import sppeec_status

workdir = tempfile.mkdtemp(prefix='sppeec_monitor_')
STATUS = os.path.join(workdir, 'status.json')
EVENTS = os.path.join(workdir, 'events.jsonl')

proc = subprocess.Popen(
    [sys.executable, os.path.join(ROOT, 'src', 'sppeec_cli.py'),
     os.path.join(ROOT, 'examples', 'module3wire.toml')],
    env=dict(os.environ, SPPEEC_STATUS=STATUS,
             SPPEEC_STATUS_EVENTS=EVENTS),
    cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True)
print('solver pid', proc.pid, '-> status file', STATUS)


In [ ]:
# -- live monitor: a persistent tqdm bar tracks overall percent (with
# the current task and frequency as its postfix), and the partial Z(f)
# sweep redraws in place -- via a display handle, only when a new
# point lands -- so nothing flickers. sppeec_status.read() adds
# _alive/_stale_s, so a crashed writer is detected rather than waited
# on forever.
import time

import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

bar = tqdm(total=100.0, desc='overall', unit='%',
           bar_format='{l_bar}{bar}| {n:.0f}/{total:.0f}% {postfix}')
plot = None
shown = 0
while True:
    try:
        d = sppeec_status.read(STATUS)
    except (FileNotFoundError, ValueError):
        time.sleep(0.3)
        continue
    ov = d['overall']['pct']
    if ov is not None:
        bar.n = min(float(ov), 100.0)
    post = {'task': d['task'].get('current') or '-'}
    if d['sweep']['current_freq'] is not None:
        post['f'] = '%g (%d/%s)' % (d['sweep']['current_freq'],
                                    d['sweep']['index'] + 1,
                                    d['sweep']['n'])
    if d['task'].get('pct') is not None:
        post['task%'] = '%.0f' % d['task']['pct']
    bar.set_postfix(post, refresh=True)
    rows = d['sweep']['results']
    if len(rows) > shown and all('R' in r for r in rows):
        shown = len(rows)
        fig, (axr, axl) = plt.subplots(1, 2, figsize=(9, 3))
        fs = [r['f'] for r in rows]
        axr.loglog(fs, [r['R'] for r in rows], 'o-')
        axr.set_xlabel('f [Hz]'); axr.set_ylabel('R [Ohm]')
        if all(r.get('L') for r in rows):
            axl.semilogx(fs, [r['L'] * 1e9 for r in rows], 'o-')
            axl.set_xlabel('f [Hz]'); axl.set_ylabel('L [nH]')
        fig.suptitle('Z(f), filling in live (%d/%s points)'
                     % (len(rows), d['sweep']['n']))
        fig.tight_layout()
        if plot is None:
            plot = display(fig, display_id=True)
        else:
            plot.update(fig)
        plt.close(fig)
    if d['state'] != 'running':
        if d['state'] == 'done':
            bar.n = 100.0
        bar.refresh()
        bar.close()
        print('final state:', d['state'])
        break
    if not d['_alive']:
        bar.close()
        print('writer process is gone without finishing')
        break
    time.sleep(0.5)
proc.wait()


In [ ]:
# -- post-mortem: the JSONL event log is append-only history. Summing
# task_end durations by task name says where the wall clock went.
import collections
import json

spent = collections.Counter()
for line in open(EVENTS):
    e = json.loads(line)
    if e['ev'] == 'task_end':
        spent[e['task']] += e['dur_s']

names = [n for n, _ in spent.most_common()]
vals = [spent[n] for n in names]
fig, ax = plt.subplots(figsize=(7, 0.4 * len(names) + 1))
ax.barh(range(len(names)), vals)
ax.set_yticks(range(len(names)), names)
ax.invert_yaxis()
ax.set_xlabel('wall seconds (summed over the run)')
ax.set_title('where the time went')
fig.tight_layout()
plt.show()


## Other ways to consume the same data

* **Terminal**: `python src/sppeec_status.py /path/status.json`
* **CLI live line**: `python src/sppeec_cli.py input.toml --status`
* **Same process** (a script driving the sweeper directly):

```python
import sppeec_status
sppeec_status.enable(callback=lambda d: my_widget.update(d))
```

The status file is written atomically and throttled (~4 Hz); poll it
from anything that can read a file. Schema and guarantees:
`docs/status_api.md`.
